In [1]:
# # For kaggle:
# !git clone https://github.com/maTiahK/CORT_SI.git
# %cd /kaggle/working/CORT_SI

# !python -m pip install -q --upgrade pip
# !python -m pip install -q -e .


# from cort_si import SI, SI_randj, gen_data
# print("CORT_SI import OK")

# Example 1: Computing p-values with Adaptive CoRT-SI

This notebook demonstrates how to use the `cort_si` package to compute selective inference p-values for Adaptive CoRT-SI.

In [2]:
import random
import sys

import numpy as np

sys.path.append('..')

from cort_si import SI_parallel_randj, SI, SI_randj, SI_parallel, gen_data

## 1. Generate Synthetic Data

In [3]:
np.random.seed(0)
random.seed(0)

p = 300
nS = 100
nT = 50
K = 5
true_beta_scale = 1.0
rho = 0.0
sigma_noise = 1.0
source_shift_sd = 0.3
covariate_shift = False
seed = 0

XS_list, YS_list, X0, Y0, _, SigmaS_list, Sigma0, beta0 = gen_data.generate_data(
    p=p,
    nS=nS,
    nT=nT,
    K=K,
    true_beta=true_beta_scale,
    rho=rho,
    sigma_noise=sigma_noise,
    source_shift_sd=source_shift_sd,
    covariate_shift=covariate_shift,
    seed=seed,
    num_info_aux=3,
    num_uninfo_aux=2
)

print(f'Number of auxiliary groups: {len(XS_list)}')
print(f'Feature dimension: {p}')
print(f'Nonzero target coefficients: {int(np.count_nonzero(beta0))}')
print(f'Source sample size: {nS}')
print(f'Target sample size: {X0.shape[0]}')

print(Y0)

Number of auxiliary groups: 5
Feature dimension: 300
Nonzero target coefficients: 5
Source sample size: 100
Target sample size: 50
[ 4.48825426e+00  6.41238151e-01 -1.26708034e+00 -2.09282546e-01
  3.44146713e-02  8.48788694e-01 -2.16513476e-01 -6.27882004e-02
  4.51965200e-01 -2.17322354e+00  1.22327989e+00 -1.32149193e+00
 -1.11223087e+00 -1.51023528e+00  1.50295936e+00  1.39608690e+00
  9.17590489e-01 -4.16493275e-01 -9.14182945e-01 -4.05463449e+00
 -8.13289635e-01 -1.20281116e+00 -1.58188230e+00 -1.27817800e+00
  3.49185966e-01 -5.48382451e-01  6.09438953e-01  3.37751895e-03
 -2.20587884e+00 -2.29117690e+00  6.52309870e-01  1.38046767e+00
 -3.48037451e-01  1.69262403e+00  2.16602318e+00  1.86154010e+00
  1.22264732e+00 -1.37648145e+00 -1.53720490e+00  9.83570288e-02
 -1.05054503e+00 -5.81313577e-02 -2.08355093e+00 -2.00609562e+00
  1.57291126e-01 -1.38111922e+00 -2.48465517e+00 -1.13255375e+00
 -2.12128913e+00  3.26832138e-01]


## 2. Set Parameters

In [4]:
lambda_sel = 0.4
lambda0 = 0.5
lambdak_list = [0.5] * len(XS_list)
T = 5
verbose = True

print(f'lambda_sel: {lambda_sel}')
print(f'lambda0: {lambda0}')
print(f'T: {T}')
print('Truncation defaults: z_min=-20, z_max=20')
print(f'verbose: {verbose}')

lambda_sel: 0.4
lambda0: 0.5
T: 5
Truncation defaults: z_min=-20, z_max=20
verbose: True


## 3. Compute p-values for All Selected Features

In [5]:
for x in range(100):
    p_values = SI_parallel_randj(
        X0,
        Y0,
        XS_list,
        YS_list,
        lambda_sel=lambda_sel,
        lambda0=lambda0,
        lambdak_list=lambdak_list,
        SigmaS_list=SigmaS_list,
        Sigma0=Sigma0,
        T=T,
        verbose=verbose,
    )

source 0, fold 1 target-only: active set = [3, 15, 37, 49, 59, 60, 90, 149, 173, 177, 216, 270]
source 0, fold 1 target+source: active set = [2]
source 0, fold 1: vote = True
source 0, fold 2 target-only: active set = [1, 2, 3, 60, 88, 278]
source 0, fold 2 target+source: active set = [2]
source 0, fold 2: vote = True
source 0, fold 3 target-only: active set = [0, 3, 15, 97, 214, 218, 254, 283, 286]
source 0, fold 3 target+source: active set = [2]
source 0, fold 3: vote = True
source 0, fold 4 target-only: active set = [2, 3, 15, 24, 149, 214, 254, 270, 283, 293]
source 0, fold 4 target+source: active set = [2]
source 0, fold 4: vote = True
source 0, fold 5 target-only: active set = [1, 14, 24, 49, 127, 149, 175, 214, 218, 252, 283, 286]
source 0, fold 5 target+source: active set = [2]
source 0, fold 5: vote = True
source 0: votes = 5/5
source 1, fold 1 target-only: active set = [3, 15, 37, 49, 59, 60, 90, 149, 173, 177, 216, 270]
source 1, fold 1 target+source: active set = []
source 

KeyboardInterrupt: 

## 4. Compute a p-value for a Random Selected Feature

This section reuses the same synthetic-data setup and parameter choice, then applies `SI_randj(...)` to one randomly chosen selected target feature.

In [ ]:
rng_seed = 11

if p_values is None:
    print('No features selected')
else:
    selected_features = [j for j, _ in p_values]
    j_rand = random.Random(rng_seed).choice(selected_features)
    random.seed(rng_seed)
    p_value_rand = SI_randj(
        X0,
        Y0,
        XS_list,
        YS_list,
        lambda_sel=lambda_sel,
        lambda0=lambda0,
        lambdak_list=lambdak_list,
        SigmaS_list=SigmaS_list,
        Sigma0=Sigma0,
        T=T,
        verbose=verbose,
    )

    print(f'Number of selected features: {len(selected_features)}')
    print(f'Selected feature set: {selected_features}')
    if p_value_rand is not None:
        print(f'Random selected feature: {j_rand}')
        print(f'true beta0[{j_rand}] = {beta0[j_rand]:.4f}')
        print(f'p-value = {p_value_rand:.4f}')
    else:
        print('No valid truncation interval for the random selected feature')

source 0, fold 1 target-only: active set = [3, 165, 170, 171]
source 0, fold 1 target+source: active set = [0]
source 0, fold 1: vote = False
source 0, fold 2 target-only: active set = [3, 73, 146]
source 0, fold 2 target+source: active set = []
source 0, fold 2: vote = False
source 0, fold 3 target-only: active set = [3, 132]
source 0, fold 3 target+source: active set = [0]
source 0, fold 3: vote = False
source 0, fold 4 target-only: active set = [0, 3, 53]
source 0, fold 4 target+source: active set = [0, 3, 162]
source 0, fold 4: vote = False
source 0, fold 5 target-only: active set = [3, 106, 132]
source 0, fold 5 target+source: active set = [0, 3]
source 0, fold 5: vote = False
source 0: votes = 0/5
source 1, fold 1 target-only: active set = [3, 165, 170, 171]
source 1, fold 1 target+source: active set = []
source 1, fold 1: vote = False
source 1, fold 2 target-only: active set = [3, 73, 146]
source 1, fold 2 target+source: active set = []
source 1, fold 2: vote = False
source 1, f